# Player Scout Engine
**StatsBomb World Cup 2022 — Player Profiling & Similarity Model**

This notebook builds a player scouting engine that:
1. Extracts player-level metrics from all WC 2022 matches
2. Computes percentile rankings across 10 performance dimensions
3. Generates radar chart profiles for any player
4. Returns the top 5 most similar players using cosine similarity

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import warnings
warnings.filterwarnings('ignore')

from statsbombpy import sb
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

print('Libraries loaded successfully.')

## 1. Load All WC 2022 Match Data

In [ ]:
# Load all WC 2022 matches (competition_id=43, season_id=106)
matches = sb.matches(competition_id=43, season_id=106)
print(f'Total matches: {len(matches)}')
matches.head(3)

In [ ]:
# Load all events across all matches
all_events = []

for match_id in matches['match_id']:
    events = sb.events(match_id=match_id)
    events['match_id'] = match_id
    all_events.append(events)

df = pd.concat(all_events, ignore_index=True)
print(f'Total events loaded: {len(df):,}')
print(f'Unique players: {df["player"].nunique()}')

## 2. Feature Engineering — Player-Level Metrics

In [ ]:
# --- SHOTS & xG ---
shots = df[df['type'] == 'Shot'].copy()

shot_stats = shots.groupby('player').agg(
    shots=('id', 'count'),
    goals=('shot_outcome', lambda x: (x == 'Goal').sum()),
    xg_total=('shot_statsbomb_xg', 'sum'),
    xg_per_shot=('shot_statsbomb_xg', 'mean')
).reset_index()

shot_stats['conversion_rate'] = shot_stats['goals'] / shot_stats['shots'].replace(0, np.nan)
shot_stats['xg_overperformance'] = shot_stats['goals'] - shot_stats['xg_total']

print(f'Shot data: {len(shot_stats)} players')

In [ ]:
# --- PASSING ---
passes = df[df['type'] == 'Pass'].copy()

# Progressive passes: move ball 10+ yards toward goal, in opponent half
def is_progressive(row):
    try:
        start_x = row['location'][0]
        end_x = row['pass_end_location'][0]
        return (end_x - start_x) >= 10 and start_x >= 60
    except:
        return False

passes['progressive'] = passes.apply(is_progressive, axis=1)

pass_stats = passes.groupby('player').agg(
    passes_attempted=('id', 'count'),
    passes_completed=('pass_outcome', lambda x: x.isna().sum()),
    progressive_passes=('progressive', 'sum'),
    key_passes=('pass_goal_assist', lambda x: x.notna().sum())
).reset_index()

pass_stats['pass_completion'] = pass_stats['passes_completed'] / pass_stats['passes_attempted'].replace(0, np.nan)

print(f'Pass data: {len(pass_stats)} players')

In [ ]:
# --- PRESSING ---
pressures = df[df['type'] == 'Pressure'].copy()

press_stats = pressures.groupby('player').agg(
    pressures_applied=('id', 'count'),
).reset_index()

# Dribbles
dribbles = df[df['type'] == 'Dribble'].copy()
dribble_stats = dribbles.groupby('player').agg(
    dribbles_attempted=('id', 'count'),
    dribbles_completed=('dribble_outcome', lambda x: (x == 'Complete').sum())
).reset_index()
dribble_stats['dribble_success'] = dribble_stats['dribbles_completed'] / dribble_stats['dribbles_attempted'].replace(0, np.nan)

print(f'Press data: {len(press_stats)} players')
print(f'Dribble data: {len(dribble_stats)} players')

In [ ]:
# --- MINUTES PLAYED (via carry/pass appearances as proxy) ---
# Use number of matches appeared in from events
appearances = df[df['player'].notna()].groupby('player').agg(
    matches_played=('match_id', 'nunique'),
    team=('team', lambda x: x.mode()[0] if len(x) > 0 else 'Unknown')
).reset_index()

print(f'Appearances data: {len(appearances)} players')

## 3. Build Master Player Profile Table

In [ ]:
# Merge all stats
player_profiles = appearances.copy()

player_profiles = player_profiles.merge(shot_stats, on='player', how='left')
player_profiles = player_profiles.merge(pass_stats, on='player', how='left')
player_profiles = player_profiles.merge(press_stats, on='player', how='left')
player_profiles = player_profiles.merge(dribble_stats, on='player', how='left')

# Fill NaN with 0
player_profiles = player_profiles.fillna(0)

# Normalize key metrics per match to make players comparable
for col in ['shots', 'goals', 'xg_total', 'progressive_passes', 'key_passes',
            'pressures_applied', 'dribbles_attempted', 'passes_attempted']:
    player_profiles[f'{col}_per_match'] = player_profiles[col] / player_profiles['matches_played'].replace(0, np.nan)

player_profiles = player_profiles.fillna(0)

# Filter to players with at least 2 matches
player_profiles = player_profiles[player_profiles['matches_played'] >= 2].reset_index(drop=True)

print(f'Player profiles built: {len(player_profiles)} players')
player_profiles[['player', 'team', 'matches_played', 'goals', 'xg_total', 'progressive_passes', 'key_passes']].head(10)

## 4. Percentile Rankings

In [ ]:
# The 10 metrics we'll use for profiling + similarity
METRICS = [
    'xg_total',
    'xg_per_shot',
    'xg_overperformance',
    'progressive_passes_per_match',
    'key_passes_per_match',
    'pass_completion',
    'pressures_applied_per_match',
    'dribble_success',
    'shots_per_match',
    'dribbles_attempted_per_match'
]

METRIC_LABELS = [
    'Total xG',
    'xG per Shot',
    'xG Overperf.',
    'Progressive\nPasses/Match',
    'Key Passes/Match',
    'Pass Completion',
    'Pressures/Match',
    'Dribble Success',
    'Shots/Match',
    'Dribbles/Match'
]

# Compute percentiles
percentile_df = player_profiles[['player', 'team', 'matches_played'] + METRICS].copy()

for metric in METRICS:
    percentile_df[f'{metric}_pct'] = percentile_df[metric].rank(pct=True) * 100

print('Percentile rankings computed.')
percentile_df[['player', 'team'] + [f'{m}_pct' for m in METRICS]].head(5)

## 5. Radar Chart — Player Profile

In [ ]:
def plot_radar(player_name, save=True):
    """Generate a radar chart for a given player."""
    
    row = percentile_df[percentile_df['player'].str.lower() == player_name.lower()]
    
    if len(row) == 0:
        # Try partial match
        row = percentile_df[percentile_df['player'].str.lower().str.contains(player_name.lower())]
        if len(row) == 0:
            print(f'Player "{player_name}" not found.')
            return
        print(f'Found: {row.iloc[0]["player"]}')
    
    row = row.iloc[0]
    full_name = row['player']
    team = row['team']
    matches = int(row['matches_played'])
    
    values = [row[f'{m}_pct'] for m in METRICS]
    
    # Radar setup
    N = len(METRICS)
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]
    values_plot = values + values[:1]
    
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    fig.patch.set_facecolor('#0d1117')
    ax.set_facecolor('#0d1117')
    
    # Grid
    ax.set_ylim(0, 100)
    ax.set_yticks([20, 40, 60, 80, 100])
    ax.set_yticklabels(['20', '40', '60', '80', '100'], color='#666666', size=7)
    ax.yaxis.set_tick_params(pad=28)
    ax.grid(color='#333333', linewidth=0.5)
    ax.spines['polar'].set_color('#333333')
    
    # Plot
    ax.plot(angles, values_plot, color='#00d4ff', linewidth=2, linestyle='solid')
    ax.fill(angles, values_plot, color='#00d4ff', alpha=0.25)
    
    # Labels
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(METRIC_LABELS, color='white', size=9)
    
    # Title
    ax.set_title(f'{full_name}\n{team}  |  {matches} matches',
                 color='white', size=14, fontweight='bold', pad=20)
    
    # Percentile dots
    for angle, val in zip(angles[:-1], values):
        ax.plot(angle, val, 'o', color='#00d4ff', markersize=5)
    
    plt.tight_layout()
    
    if save:
        fname = f'radar_{full_name.replace(" ", "_").replace(".", "")}.png'
        plt.savefig(fname, dpi=150, bbox_inches='tight', facecolor='#0d1117')
        print(f'Saved: {fname}')
    
    plt.show()
    return full_name

# Example
plot_radar('Lionel Messi')

## 6. Player Similarity Model — Top 5 Most Similar Players

In [ ]:
# Build feature matrix for similarity
feature_cols = METRICS
X = player_profiles.set_index('player')[feature_cols].copy()

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, index=X.index, columns=feature_cols)

# Compute cosine similarity matrix
sim_matrix = cosine_similarity(X_scaled_df)
sim_df = pd.DataFrame(sim_matrix, index=X_scaled_df.index, columns=X_scaled_df.index)

print(f'Similarity matrix built: {sim_df.shape}')

In [ ]:
def find_similar_players(player_name, n=5):
    """Return top N most similar players."""
    
    # Match player name
    matches_found = [p for p in sim_df.index if player_name.lower() in p.lower()]
    if not matches_found:
        print(f'Player "{player_name}" not found.')
        return
    
    full_name = matches_found[0]
    team = player_profiles[player_profiles['player'] == full_name]['team'].values[0]
    
    similar = sim_df[full_name].drop(full_name).sort_values(ascending=False).head(n)
    
    print(f'\n🔍 Top {n} players most similar to {full_name} ({team}):\n')
    print(f'{"Rank":<6}{"Player":<30}{"Team":<25}{"Similarity"}')
    print('-' * 75)
    
    for i, (name, score) in enumerate(similar.items(), 1):
        player_team = player_profiles[player_profiles['player'] == name]['team'].values
        player_team = player_team[0] if len(player_team) > 0 else 'Unknown'
        print(f'{i:<6}{name:<30}{player_team:<25}{score:.3f}')
    
    return similar

find_similar_players('Messi')

## 7. Full Scouting Report — Radar + Similarity Combined

In [ ]:
def scouting_report(player_name):
    """Full scouting report: radar chart + similar players + key stats."""
    
    # Find player
    matches_found = [p for p in percentile_df['player'].values if player_name.lower() in p.lower()]
    if not matches_found:
        print(f'Player "{player_name}" not found.')
        return
    
    full_name = matches_found[0]
    row = percentile_df[percentile_df['player'] == full_name].iloc[0]
    profile_row = player_profiles[player_profiles['player'] == full_name].iloc[0]
    
    print('=' * 60)
    print(f'  SCOUTING REPORT: {full_name}')
    print(f'  Team: {row["team"]}  |  Matches: {int(row["matches_played"])}')
    print('=' * 60)
    
    print('\n📊 KEY STATS\n')
    stats_to_show = [
        ('Goals', 'goals'),
        ('Total xG', 'xg_total'),
        ('xG Overperformance', 'xg_overperformance'),
        ('Shots', 'shots'),
        ('Progressive Passes', 'progressive_passes'),
        ('Key Passes', 'key_passes'),
        ('Pass Completion %', 'pass_completion'),
        ('Pressures Applied', 'pressures_applied'),
        ('Dribbles Completed', 'dribbles_completed'),
    ]
    
    for label, col in stats_to_show:
        if col in profile_row:
            val = profile_row[col]
            if col == 'pass_completion':
                print(f'  {label:<25} {val*100:.1f}%')
            else:
                print(f'  {label:<25} {val:.2f}')
    
    print('\n📈 PERCENTILE RANKINGS (vs all WC 2022 players)\n')
    for metric, label in zip(METRICS, METRIC_LABELS):
        pct = row[f'{metric}_pct']
        bar = '█' * int(pct / 10) + '░' * (10 - int(pct / 10))
        clean_label = label.replace('\n', ' ')
        print(f'  {clean_label:<25} {bar}  {pct:.0f}th')
    
    # Radar
    print('\n')
    plot_radar(player_name, save=True)
    
    # Similar players
    find_similar_players(player_name)


# Run full report
scouting_report('Messi')

In [ ]:
# Try another player
scouting_report('Mbappe')

In [ ]:
# Try another player
scouting_report('Bukayo Saka')

## 8. Top Players by Metric — Tournament Rankings

In [ ]:
def top_players(metric, n=10, label=None):
    """Show top N players for a given metric."""
    label = label or metric
    top = player_profiles[['player', 'team', metric]].sort_values(metric, ascending=False).head(n)
    print(f'\nTop {n} players by {label}:\n')
    print(f'{"Rank":<6}{"Player":<30}{"Team":<25}{label}')
    print('-' * 75)
    for i, (_, r) in enumerate(top.iterrows(), 1):
        print(f'{i:<6}{r["player"]:<30}{r["team"]:<25}{r[metric]:.3f}')

top_players('xg_total', label='Total xG')
top_players('progressive_passes_per_match', label='Progressive Passes/Match')
top_players('pressures_applied_per_match', label='Pressures/Match')